In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE

# ── Apply all Task 2 cleaning steps ──
df = pd.read_csv('churnguard_data.csv')
df = df.drop('customerID', axis=1)
df = df.drop_duplicates()
df['gender'] = df['gender'].str.strip()
df['PaymentMethod'] = df['PaymentMethod'].str.strip()
df['Churn'] = df['Churn'].str.strip().str.title()
df['PhoneService'] = df['PhoneService'].str.strip().str.title()
df['PaperlessBilling'] = df['PaperlessBilling'].str.strip().str.title()

contract_map = {
    'Month-to-month': 'Month-to-month', 'month to month': 'Month-to-month',
    'month-to-month': 'Month-to-month', 'Monthly': 'Month-to-month',
    'One year': 'One year', 'One Year': 'One year',
    'one year': 'One year', '1 year': 'One year',
    'Two year': 'Two year', 'Two Year': 'Two year',
    'two year': 'Two year', '2 year': 'Two year',
}
df['Contract'] = df['Contract'].map(contract_map)

internet_map = {
    'DSL': 'DSL', 'dsl': 'DSL',
    'Fiber optic': 'Fiber optic', 'Fibre optic': 'Fiber optic',
    'FiberOptic': 'Fiber optic', 'fiber optic': 'Fiber optic',
    'No': 'No', 'None': 'No', 'none': 'No',
}
df['InternetService'] = df['InternetService'].map(internet_map)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df[df['tenure'] > 0]
df = df[(df['MonthlyCharges'] >= 10) & (df['MonthlyCharges'] <= 200)]
df['MonthlyCharges'] = df['MonthlyCharges'].fillna(df['MonthlyCharges'].mean())
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].mean())
df['tenure'] = df['tenure'].fillna(int(df['tenure'].median()))

# ── Encode target ──
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# ── Encode categorical columns ──
cat_cols = ['gender', 'PhoneService', 'InternetService',
            'Contract', 'PaperlessBilling', 'PaymentMethod']
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# ── Separate X and y ──
X = df.drop('Churn', axis=1)
y = df['Churn']

# ── Train test split ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print("=" * 60)
print("BEFORE IMPROVEMENTS (Task 3 baseline)")
print("=" * 60)
model_base = LogisticRegression(max_iter=1000)
model_base.fit(X_train, y_train)
y_pred_base = model_base.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred_base):.4f}")
print(classification_report(y_test, y_pred_base,
      target_names=['Stay', 'Churn']))

# ── Improvement 1: StandardScaler ──
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ── Improvement 2: SMOTE on training data only ──
print("Class distribution before SMOTE:")
print(f"Stay={sum(y_train==0)}, Churn={sum(y_train==1)}")

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train_scaled, y_train)

print("\nClass distribution after SMOTE:")
print(f"Stay={sum(y_train_resampled==0)}, Churn={sum(y_train_resampled==1)}")

# ── Improvement 3: class_weight + higher max_iter ──
print("\n" + "=" * 60)
print("AFTER IMPROVEMENTS (Task 5)")
print("=" * 60)
model_improved = LogisticRegression(
    max_iter=5000,
    class_weight='balanced',
    random_state=42
)
model_improved.fit(X_train_resampled, y_train_resampled)
y_pred_improved = model_improved.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred_improved):.4f}")
print(classification_report(y_test, y_pred_improved,
      target_names=['Stay', 'Churn']))

# ── Improvement 4: Custom threshold ──
print("=" * 60)
print("WITH CUSTOM THRESHOLD (0.3)")
print("=" * 60)
probabilities = model_improved.predict_proba(X_test_scaled)[:, 1]
y_pred_threshold = (probabilities > 0.3).astype(int)
print(f"Accuracy: {accuracy_score(y_test, y_pred_threshold):.4f}")
print(classification_report(y_test, y_pred_threshold,
      target_names=['Stay', 'Churn']))

# ── Comparison Summary ──
print("=" * 60)
print("IMPROVEMENT SUMMARY")
print("=" * 60)
from sklearn.metrics import recall_score
recall_base     = recall_score(y_test, y_pred_base, pos_label=1)
recall_improved = recall_score(y_test, y_pred_improved, pos_label=1)
recall_threshold= recall_score(y_test, y_pred_threshold, pos_label=1)

print(f"Task 3 baseline  → Churn Recall: {recall_base:.2f}")
print(f"Task 5 improved  → Churn Recall: {recall_improved:.2f}")
print(f"Task 5 threshold → Churn Recall: {recall_threshold:.2f}")

BEFORE IMPROVEMENTS (Task 3 baseline)
Accuracy: 0.6954
              precision    recall  f1-score   support

        Stay       0.74      0.87      0.80       122
       Churn       0.48      0.29      0.36        52

    accuracy                           0.70       174
   macro avg       0.61      0.58      0.58       174
weighted avg       0.66      0.70      0.67       174

Class distribution before SMOTE:
Stay=471, Churn=222

Class distribution after SMOTE:
Stay=471, Churn=471

AFTER IMPROVEMENTS (Task 5)
Accuracy: 0.6322
              precision    recall  f1-score   support

        Stay       0.85      0.58      0.69       122
       Churn       0.43      0.75      0.55        52

    accuracy                           0.63       174
   macro avg       0.64      0.67      0.62       174
weighted avg       0.72      0.63      0.65       174

WITH CUSTOM THRESHOLD (0.3)
Accuracy: 0.6264
              precision    recall  f1-score   support

        Stay       0.84      0.57      

c:\Users\vemur\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
